In [1]:
"""
This considers the checkout of past records, and using them, based on the latest itemset in the cart(and unique items in the latest cart), we show recommendations.

Then if the recommendation count is not full, consider the unique items in the entire history and recommend their consequents.

"""

'\nThis considers the checkout of past records, and using them, based on the latest itemset in the cart(and unique items in the latest cart), we show recommendations.\n\nThen if the recommendation count is not full, consider the unique items in the entire history and recommend their consequents.\n\n'

# Stats of best item to show to users of "each cart time stamps" or "unique" items.

In [2]:
import csv
import pandas as pd

csv_file_for_cart_timestamp = 'cart_timestamp.csv'
def history_update(historyy,time, new_1_item):
        final_latest_item_list = []
        if len(list(historyy)) >= 1:
                latest_timestamp = list(historyy)[-1]
                latest_item_list = historyy[latest_timestamp]
                for i in latest_item_list:
                        final_latest_item_list.append(i)      
        final_latest_item_list.append(new_1_item)
        historyy.update({time: final_latest_item_list})
        # Need to uncomment below
        # with open(csv_file_for_cart_timestamp, 'a', newline='') as csvfile:
        #         writer = csv.writer(csv_file_for_cart_timestamp)
        #         # Add new rows
        #         writer.writerow([f"Time:=0940,Milk"])

def history_delete(historyy,time, new_to_delete_item):
    final_latest_item_list = []
    if len(list(historyy)) >= 1:
            latest_timestamp = list(historyy)[-1]
            latest_item_list = historyy[latest_timestamp]
            for i in latest_item_list:
                    final_latest_item_list.append(i)      
    final_latest_item_list.remove(new_to_delete_item)
    historyy.update({time: final_latest_item_list})
        
def getting_items_in_at_needed_real_timestamp(historyy,timestamp):
    return historyy[timestamp]

        


    



In [3]:
from mlxtend.frequent_patterns import apriori,association_rules
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

def finding_association_rules(csv_file):
    data = {
        "Order":[

        ]
    }
    df_temp = pd.read_csv(csv_file)


    
    for i in range(0,len(df_temp)):
        value_ = df_temp.Order[i].split(',')
        data['Order'].append(value_)
    data
    
    df = pd.DataFrame(data)

    te = TransactionEncoder()
    te_ary = te.fit_transform(df['Order'])

    df_trans = pd.DataFrame(te_ary, columns = te.columns_)

    frequent_itemsets = apriori(df_trans, min_support=0.2, use_colnames=True)

    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0.5)
    
    return frequent_itemsets, rules

csv_file = 'orders.csv'
frequent_itemsets, rules = finding_association_rules(csv_file)

In [4]:
historyy = dict()
historyy
history_update(historyy, "Time:=0920","Coffee")
history_update(historyy, "Time:=0930","Tea")
historyy
history_delete(historyy, "Time:=0945", "Coffee")
historyy
history_update(historyy, "Time:=0950","Ginger")
historyy

{'Time:=0920': ['Coffee'],
 'Time:=0930': ['Coffee', 'Tea'],
 'Time:=0945': ['Tea'],
 'Time:=0950': ['Tea', 'Ginger']}

In [5]:
finalPredictionDf = {
        'LatestCartItems/uniqueItemInLatestCart' : [],
        'antecedents': [],
        'consequents': [],
        'support': [] ,
        'confidence': []
        }

finalPredictionDf = pd.DataFrame(finalPredictionDf)
finalPredictionDf

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence


In [6]:
def predictt2(rules, new_order):
    new_order_ = frozenset(new_order)
    recommendations = rules[rules['antecedents'] == new_order_]
    return recommendations[['antecedents','consequents','support' ,'confidence']]

In [7]:
# latest history items
def getting_items_in_at_needed_timestamp(historyy,needed_time_stamp_position):
    if len(historyy) != 0:
        return set(historyy[list(historyy)[needed_time_stamp_position]])
    

itemset_in_latest_cart = getting_items_in_at_needed_timestamp(historyy,-1)
predictions = predictt2(rules, itemset_in_latest_cart)
predictions.sort_values('confidence', ascending=False)
predictions['LatestCartItems/uniqueItemInLatestCart'] = 'LatestCartItems'
finalPredictionDf = pd.concat([finalPredictionDf, predictions])
finalPredictionDf


# Uncomment below to grab the df at where it has predictions for latest cart.
# predictions_for_latest_cart_items_only = finalPredictionDf

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence
9,LatestCartItems,"(Ginger, Tea)",(cinnamon),0.5,0.8


In [8]:
unique_items_in_lastest_cart = []
for i in getting_items_in_at_needed_timestamp(historyy,-1):
    unique_items_in_lastest_cart.append(i)
unique_items_in_lastest_cart = list(dict.fromkeys(unique_items_in_lastest_cart))
unique_items_in_lastest_cart

['Ginger', 'Tea']

In [9]:
for i in unique_items_in_lastest_cart:
    items = []
    items.append(i)
    predictions = predictt2(rules, set(items))
    predictions.sort_values('confidence', ascending=False)
    predictions['LatestCartItems/uniqueItemInLatestCart'] = 'uniqueItemInLatestCart'
    finalPredictionDf = pd.concat([finalPredictionDf, predictions])
finalPredictionDf

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence
9,LatestCartItems,"(Ginger, Tea)",(cinnamon),0.500,0.8
2,uniqueItemInLatestCart,(Ginger),(Tea),0.625,1.0
4,uniqueItemInLatestCart,(Ginger),(cinnamon),0.500,0.8
11,uniqueItemInLatestCart,(Ginger),"(Tea, cinnamon)",0.500,0.8
3,uniqueItemInLatestCart,(Tea),(Ginger),0.625,1.0
7,uniqueItemInLatestCart,(Tea),(cinnamon),0.500,0.8
13,uniqueItemInLatestCart,(Tea),"(Ginger, cinnamon)",0.500,0.8


sort in support, ~~then~~ and sort in confidence

In [10]:
finalPredictionDf
sorted_in_support = finalPredictionDf.sort_values('support', ascending=False).drop_duplicates()
sorted_in_support
sorted_in_confidence = finalPredictionDf.sort_values('confidence', ascending=False).drop_duplicates()
sorted_in_confidence

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence
2,uniqueItemInLatestCart,(Ginger),(Tea),0.625,1.0
3,uniqueItemInLatestCart,(Tea),(Ginger),0.625,1.0
9,LatestCartItems,"(Ginger, Tea)",(cinnamon),0.500,0.8
4,uniqueItemInLatestCart,(Ginger),(cinnamon),0.500,0.8
11,uniqueItemInLatestCart,(Ginger),"(Tea, cinnamon)",0.500,0.8
7,uniqueItemInLatestCart,(Tea),(cinnamon),0.500,0.8
13,uniqueItemInLatestCart,(Tea),"(Ginger, cinnamon)",0.500,0.8


sort in the order of 
    confidence, then
    support

In [11]:
# Let sort from confidence and then support
finalPredictionDf
sorted_in_confidence = finalPredictionDf.sort_values('confidence', ascending=False).drop_duplicates()
sorted_in_confidence
sorted_in_confidence_support = sorted_in_confidence.sort_values('support', ascending=False).drop_duplicates()
sorted_in_confidence_support

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence
2,uniqueItemInLatestCart,(Ginger),(Tea),0.625,1.0
3,uniqueItemInLatestCart,(Tea),(Ginger),0.625,1.0
9,LatestCartItems,"(Ginger, Tea)",(cinnamon),0.500,0.8
4,uniqueItemInLatestCart,(Ginger),(cinnamon),0.500,0.8
11,uniqueItemInLatestCart,(Ginger),"(Tea, cinnamon)",0.500,0.8
7,uniqueItemInLatestCart,(Tea),(cinnamon),0.500,0.8
13,uniqueItemInLatestCart,(Tea),"(Ginger, cinnamon)",0.500,0.8


sort in the order of
    support, then
    confidence

In [12]:
finalPredictionDf
sorted_in_support = finalPredictionDf.sort_values('support', ascending=False).drop_duplicates()
sorted_in_support
sorted_in_support_confidence = sorted_in_support.sort_values('confidence', ascending=False).drop_duplicates()
sorted_in_support_confidence

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence
2,uniqueItemInLatestCart,(Ginger),(Tea),0.625,1.0
3,uniqueItemInLatestCart,(Tea),(Ginger),0.625,1.0
9,LatestCartItems,"(Ginger, Tea)",(cinnamon),0.500,0.8
4,uniqueItemInLatestCart,(Ginger),(cinnamon),0.500,0.8
11,uniqueItemInLatestCart,(Ginger),"(Tea, cinnamon)",0.500,0.8
7,uniqueItemInLatestCart,(Tea),(cinnamon),0.500,0.8
13,uniqueItemInLatestCart,(Tea),"(Ginger, cinnamon)",0.500,0.8


Ig it is better 
    just not doing as "sorted_in_support", and then "sorted_in_support_confidence", and take "sorted_in_support_confidence" as the finalPredictionDf
                            but do as 
    (A) and/or (B)
        
        (A)
            take 50% from the "sorted_in_support" and take 50% from the "sorted_in_support_confidence" and take each's those 50 and combine as 1 and take it as the "finalPredictionDf"
        (B)
            take 40% from ones which are both higher with support and AT THE SAME TIME, and take 30% from "sorted_in_support" and 30% from "sorted_in_support_confidence" and take it as the "finalPredictionDf"

Top 50 percent from each

In [13]:
finalPredictionDf
sorted_in_support = finalPredictionDf.sort_values('support', ascending=False).drop_duplicates()
sorted_in_support

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence
2,uniqueItemInLatestCart,(Ginger),(Tea),0.625,1.0
3,uniqueItemInLatestCart,(Tea),(Ginger),0.625,1.0
9,LatestCartItems,"(Ginger, Tea)",(cinnamon),0.500,0.8
4,uniqueItemInLatestCart,(Ginger),(cinnamon),0.500,0.8
11,uniqueItemInLatestCart,(Ginger),"(Tea, cinnamon)",0.500,0.8
7,uniqueItemInLatestCart,(Tea),(cinnamon),0.500,0.8
13,uniqueItemInLatestCart,(Tea),"(Ginger, cinnamon)",0.500,0.8


In [14]:

sorted_in_confidence = finalPredictionDf.sort_values('confidence', ascending=False).drop_duplicates()
sorted_in_confidence

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence
2,uniqueItemInLatestCart,(Ginger),(Tea),0.625,1.0
3,uniqueItemInLatestCart,(Tea),(Ginger),0.625,1.0
9,LatestCartItems,"(Ginger, Tea)",(cinnamon),0.500,0.8
4,uniqueItemInLatestCart,(Ginger),(cinnamon),0.500,0.8
11,uniqueItemInLatestCart,(Ginger),"(Tea, cinnamon)",0.500,0.8
7,uniqueItemInLatestCart,(Tea),(cinnamon),0.500,0.8
13,uniqueItemInLatestCart,(Tea),"(Ginger, cinnamon)",0.500,0.8


In [15]:
length_of_sorted_in_support = int(len(sorted_in_support) / 2)
sorted_in_support_50_percent = sorted_in_support.head(length_of_sorted_in_support)
sorted_in_support_50_percent

length_of_sorted_in_confidence = int(len(sorted_in_confidence) / 2)
sorted_in_confidence_50_percent = sorted_in_confidence.head(length_of_sorted_in_confidence)
sorted_in_confidence_50_percent


Top50PercentFromEach  = pd.concat([sorted_in_support_50_percent, sorted_in_confidence_50_percent])

# In case we missed any important ones during this 50 percent takings. We can use either of the sorted_in_support, sorted_in_confidence to get these left ones. So after we do this, we remove duplicates..so no problem
Top50PercentFromEach = pd.concat([Top50PercentFromEach, sorted_in_confidence])

#Remove the duplicates which came during adding left ones.
Top50PercentFromEach = Top50PercentFromEach.drop_duplicates()
Top50PercentFromEach

,LatestCartItems/uniqueItemInLatestCart,antecedents,consequents,support,confidence
2,uniqueItemInLatestCart,(Ginger),(Tea),0.625,1.0
3,uniqueItemInLatestCart,(Tea),(Ginger),0.625,1.0
9,LatestCartItems,"(Ginger, Tea)",(cinnamon),0.500,0.8
4,uniqueItemInLatestCart,(Ginger),(cinnamon),0.500,0.8
11,uniqueItemInLatestCart,(Ginger),"(Tea, cinnamon)",0.500,0.8
7,uniqueItemInLatestCart,(Tea),(cinnamon),0.500,0.8
13,uniqueItemInLatestCart,(Tea),"(Ginger, cinnamon)",0.500,0.8


In [16]:
#In above, for the Top 50 percent thing, we can consider 50 percent of each as below too btw.

"""    
    # sorted_in_support 
    # then (not "and"!) 
    # 50 percent of sorted_in_support_confidence 
"""

    # and not just

"""
    # 50 percent of sorted_in_support
    # and (not "then"!) 
    # 50 percent of sorted_in_confidence
"""

'\n    # 50 percent of sorted_in_support\n    # and (not "then"!) \n    # 50 percent of sorted_in_confidence\n'

In [17]:
            # from either support_then_confidence, or confidence_then_support, or Top50PercentFromEach, the dataframe to follow the next steps is below

# dataFrameForNextSteps = sorted_in_support_confidence
# dataFrameForNextSteps = sorted_in_confidence_support
dataFrameForNextSteps = Top50PercentFromEach

Removing predictions which the user has already got in cart

In [18]:
def unpack_frozenset_items(k):
    listt = []
    for i in k:
        listt.append(i)
    return listt

show_these_to_user = dataFrameForNextSteps['consequents']
final_recommendations = []
for i in list(show_these_to_user):
    frozenset_items = unpack_frozenset_items(i)
    for j in frozenset_items:
        final_recommendations.append(j)

final_recommendations = list(dict.fromkeys(final_recommendations))


for i in unique_items_in_lastest_cart:
    if i in final_recommendations:
        final_recommendations.remove(i)

print("Show user with", final_recommendations)
print("Where the cart history is", [i for i in historyy.values()])

Show user with ['cinnamon']
Where the cart history is [['Coffee'], ['Coffee', 'Tea'], ['Tea'], ['Tea', 'Ginger']]


In [19]:
print("If the receommendations are not enough, show below too")

If the receommendations are not enough, show below too


In [20]:
finalPredictionDf_if_recom_is_not_full = {
        'antecedents': [],
        'consequents': [],
        'support': [] ,
        'confidence': []
        }

finalPredictionDf_if_recom_is_not_full = pd.DataFrame(finalPredictionDf_if_recom_is_not_full)
finalPredictionDf_if_recom_is_not_full

,antecedents,consequents,support,confidence


In [21]:
unique_items = []
for i in historyy.values():
    for j in i:
        unique_items.append(j)
unique_items = list(dict.fromkeys(unique_items))
unique_items

['Coffee', 'Tea', 'Ginger']

In [22]:
for i in unique_items:
    items = []
    items.append(i)
    predictions = predictt2(rules, set(items))
    predictions.sort_values('confidence', ascending=False)
    finalPredictionDf_if_recom_is_not_full = pd.concat([finalPredictionDf_if_recom_is_not_full, predictions])
finalPredictionDf_if_recom_is_not_full    


# Uncomment below to grab the df at where it has predictions for all unique items only.
# predictions_for_unique_items_only = finalPredictionDf_if_recom_is_not_full


,antecedents,consequents,support,confidence
0,(Coffee),(Biscuit),0.375,1.0
3,(Tea),(Ginger),0.625,1.0
7,(Tea),(cinnamon),0.500,0.8
13,(Tea),"(Ginger, cinnamon)",0.500,0.8
2,(Ginger),(Tea),0.625,1.0
4,(Ginger),(cinnamon),0.500,0.8
11,(Ginger),"(Tea, cinnamon)",0.500,0.8


sort by confidence, then support

In [23]:
sorted_in_confidence_if_not_full = finalPredictionDf_if_recom_is_not_full.sort_values('confidence', ascending=False).drop_duplicates()
sorted_in_confidence_support_if_not_full = sorted_in_confidence_if_not_full.sort_values('support', ascending=False).drop_duplicates()
sorted_in_confidence_support_if_not_full


,antecedents,consequents,support,confidence
3,(Tea),(Ginger),0.625,1.0
2,(Ginger),(Tea),0.625,1.0
7,(Tea),(cinnamon),0.500,0.8
13,(Tea),"(Ginger, cinnamon)",0.500,0.8
4,(Ginger),(cinnamon),0.500,0.8
11,(Ginger),"(Tea, cinnamon)",0.500,0.8
0,(Coffee),(Biscuit),0.375,1.0


In [24]:
# Similar to what we did for the latest cart items and unique items in the cart, we can do that 50 percent thing to this too btw. But for now, let's not do.

show_these_to_user_if_not_full = sorted_in_confidence_support_if_not_full['consequents']
final_recommendations_if_not_full = []
for i in list(show_these_to_user_if_not_full):
    frozenset_items = unpack_frozenset_items(i)
    for j in frozenset_items:
        final_recommendations_if_not_full.append(j)

final_recommendations_if_not_full = list(dict.fromkeys(final_recommendations_if_not_full))
print("final_recommendations_if_not_full: ",final_recommendations_if_not_full)


for i in unique_items_in_lastest_cart:
    if i in final_recommendations_if_not_full:
        final_recommendations_if_not_full.remove(i)

print("Show user with", final_recommendations_if_not_full)
print("Where the cart history is", [i for i in historyy.values()])

final_recommendations_if_not_full:  ['Ginger', 'Tea', 'cinnamon', 'Biscuit']
Show user with ['cinnamon', 'Biscuit']
Where the cart history is [['Coffee'], ['Coffee', 'Tea'], ['Tea'], ['Tea', 'Ginger']]


In [25]:
print("So if the recommendations are not full with only latest cart items, we show below too")
print(final_recommendations_if_not_full)

So if the recommendations are not full with only latest cart items, we show below too
['cinnamon', 'Biscuit']


In [26]:
finall = []

In [27]:
print("So the final list of recommendations would be")
finall = final_recommendations + final_recommendations_if_not_full + finall   #the RHS finall is the rules for cold start problem
finall = list(dict.fromkeys(finall))
print(finall)

So the final list of recommendations would be
['cinnamon', 'Biscuit']


Order
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"

In [28]:
print("But if need to get a percentage from each of the 2 recommendation type of lists, we do below")
print("Before doing that, we need to remove the ones which are at 'final_recommendations_if_not_full' which are also in 'final_recommendations' from  final_recommendations_if_not_full")
print("Yes this is kinda similar to the removing thing we did to the recommendations which are already in the latest cart.")

print(final_recommendations)
print(len(final_recommendations))
print(final_recommendations_if_not_full)
print(len(final_recommendations_if_not_full))

for i in final_recommendations:
    if i in final_recommendations_if_not_full:
        final_recommendations_if_not_full.remove(i)

print(final_recommendations)
print(len(final_recommendations))
print(final_recommendations_if_not_full)
print(len(final_recommendations_if_not_full))

length_of_recommendations = len(final_recommendations) + len(final_recommendations_if_not_full)


#Let's build this percentage thing later...

But if need to get a percentage from each of the 2 recommendation type of lists, we do below
Before doing that, we need to remove the ones which are at 'final_recommendations_if_not_full' which are also in 'final_recommendations' from  final_recommendations_if_not_full
Yes this is kinda similar to the removing thing we did to the recommendations which are already in the latest cart.
['cinnamon']
1
['cinnamon', 'Biscuit']
2
['cinnamon']
1
['Biscuit']
1


In [29]:
print("Therefore, with upto now the history of cart details being")
print(historyy)
print("finally we show to the user with")
print(finall)
print("for the dataset with")
pd.read_csv(csv_file)

Therefore, with upto now the history of cart details being
{'Time:=0920': ['Coffee'], 'Time:=0930': ['Coffee', 'Tea'], 'Time:=0945': ['Tea'], 'Time:=0950': ['Tea', 'Ginger']}
finally we show to the user with
['cinnamon', 'Biscuit']
for the dataset with


,Order
0,"Tea,Ginger,cinnamon"
1,"Tea,Ginger,cinnamon"
2,"Tea,Ginger,cinnamon"
3,"Tea,Ginger,cinnamon"
4,"Tea,Ginger,clove"
5,"Coffee,Biscuit"
6,"Coffee,Biscuit"
7,"Coffee,Biscuit"


# Test out with these difference scenarios

In [30]:
"""
latest cart has Tea,Ginger. But we don't have rules for it. Therefore, we find for the just before latest cart which is Tea and it has Ginger. But since our latest cart has Ginger, we don't show it.
And with Coffee being in the history, we find Coffee->Biscuit. Therefore we show only Biscuit.
Order 
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"


Order
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,clove"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"



cinnamon and kurudu have the same priority but with the default order stuff, cinnamon comes to first.
Order
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"




kurudu comes fore cinnamon, which is cool!
Order
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger,kurudu"
"Tea,Ginger,kurudu"
"Tea,Ginger,cinnamon,kurudu"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"



In this, the recommendations from latest cart and unique items in latest cart is 3, and from all unique items in the history it is 4. Note: This 4 includes the before 3 too. Reason is, the dataset is small noh.

Order
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"


Order
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"


Order
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"


Order
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal,clove"
"Tea,Ginger,cinnamon,kurudu,enasal,clove"
"Tea,Ginger,cinnamon,kurudu,enasal,clove"
"Tea,Ginger,clove"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"



enasal is showed, but clove. And kurudu comes fore cinnamon
Order
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,clove"
"Tea,Ginger,kurudu,clove"
"Tea,Ginger,clove"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"



Clove is showed but enasal.
Order
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,clove"
"Tea,Ginger,cinnamon,kurudu,clove"
"Tea,Ginger,cinnamon,kurudu,clove"
"Tea,Ginger,clove"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"


Both clove and enasal is showed. Reason in earleri dataset (ex: enasal) was no showed is cuz at aprior, we gave min_support as 0.2 noh. And reason now it shows is, we added these 2. '  "Tea,Ginger,cinnamon,kurudu,enasal"  "Tea,Ginger,cinnamon,kurudu,clove"  '
Order
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,enasal"
"Tea,Ginger,cinnamon,kurudu,clove"
"Tea,Ginger,cinnamon,kurudu,clove"
"Tea,Ginger,cinnamon,kurudu,clove"
"Tea,Ginger,cinnamon,kurudu,clove"
"Tea,Ginger,clove"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"
"Coffee,Chocolate"

"""

'\nlatest cart has Tea,Ginger. But we don\'t have rules for it. Therefore, we find for the just before latest cart which is Tea and it has Ginger. But since our latest cart has Ginger, we don\'t show it.\nAnd with Coffee being in the history, we find Coffee->Biscuit. Therefore we show only Biscuit.\nOrder \n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n\n\nOrder\n"Tea,Ginger,cinnamon"\n"Tea,Ginger,cinnamon"\n"Tea,Ginger,cinnamon"\n"Tea,Ginger,cinnamon"\n"Tea,Ginger,clove"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n\n\n\ncinnamon and kurudu have the same priority but with the default order stuff, cinnamon comes to first.\nOrder\n"Tea,Ginger,cinnamon,kurudu"\n"Tea,Ginger,cinnamon,kurudu"\n"Tea,Ginger,cinnamon,kurudu"\n"Tea,Ginger,cinnamon,kurudu"\n"Tea,Ginger,cinnamon,kurudu"\n"Tea,Ginger,cinnamon,kurudu"\n"Tea,Ginger"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,

In [31]:
"""
---binary kind of comparison
High-High
High-Less
Less-High
Less-Less

Tea->Ginger    : COMPARED TO THAT OF "Coffee->Biscuit",     High confidence, High support,
Order
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"

Tea->Ginger    : COMPARED TO THAT OF "Coffee->Biscuit",     Less confidence, Less support,
Order
"Tea,Ginger"
"Tea,Ginger"
"Tea"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"

Tea->Ginger    : COMPARED TO THAT OF "Coffee->Biscuit",     High confidence, Less support,
Order
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"

Tea->Ginger    : COMPARED TO THAT OF "Coffee->Biscuit",     Less confidence, High support,
Order
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Arabic tea"
"Tea,Persian"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"






The most basic thing
Order
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"







Below compares around 4 stuff
Order
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Ginger"
"Tea,Persian"
"Tea,Arabic"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"





Very important success I got: The fact that appending rules to the dataframe were done as "latest items", "items before latest items", ... , "each unique item in the cart" is good as, if the supports or confidences are equal, the top comes to the ones of that "latest items", "items before latest items", ... , "each unique item in the cart" itself.
It was as this. The cart was as 
                                {'0920': ['Coffee'],
                                '0930': ['Coffee', 'Tea'],
                                '0945': ['Tea'],
                                '0950': ['Tea', 'Ginger']}
So when see the below rules, the predictions came as Tea_Ginger->cinnamon, came on top of Coffee->Biscuit
    Also, Tea_Ginger->cinnamon, came on top of Tea->Ginger
Order
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"





# With my "2 Several stable yet complex versions" the results did not come as expected. But with my "3 Using '2 Several...', a from scratch version", results came perfectly
Order
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Tea,Ginger,cinnamon"
"Coffee,Biscuit"
"Coffee,Biscuit"
"Coffee,Biscuit"



"""



'\n---binary kind of comparison\nHigh-High\nHigh-Less\nLess-High\nLess-Less\n\nTea->Ginger    : COMPARED TO THAT OF "Coffee->Biscuit",     High confidence, High support,\nOrder\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n\nTea->Ginger    : COMPARED TO THAT OF "Coffee->Biscuit",     Less confidence, Less support,\nOrder\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n\nTea->Ginger    : COMPARED TO THAT OF "Coffee->Biscuit",     High confidence, Less support,\nOrder\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n\nTea->Ginger    : COMPARED TO THAT OF "Coffee->Biscuit",     Less confidence, High support,\nOrder\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Ginger"\n"Tea,Arabic tea"\n"Tea,Persian"\n"Coffee,Biscuit"\n"Coffee,Biscuit"\n"Coffee,Biscuit